In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import Imputer, PolynomialFeatures, PowerTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.cross_validation import train_test_split
from sklearn.linear_model import Ridge, RidgeCV, LassoCV
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, export_graphviz, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVC

In [ ]:
year2015 = pd.read_excel('2015 FE Guide.xlsx')
year2016 = pd.read_excel('2016 FE Guide.xlsx')
year2017 = pd.read_excel('2017 FE Guide.xlsx')
year2018 = pd.read_excel('2018 FE Guide.xlsx')

# Task 1: Linear Models

## Preprocessing

In [ ]:
data = pd.concat([year2015, year2016, year2017, year2018])
data.shape

In [ ]:
data = data.dropna(axis=1,how='all')
data = data.dropna(thresh=round(0.75*data.shape[0]), axis=1)
data.shape

In [ ]:
target = data.loc[:,['Model Year','Comb Unrd Adj FE - Conventional Fuel']]
target.shape

In [ ]:
target_train = target[target['Model Year']!=2018]
target_train = target_train.drop(columns = ['Model Year'])
print("shape of the train target:{}".format(target_train.shape))

In [ ]:
target_test = target[target['Model Year']==2018]
target_test = target_test.drop(columns = ['Model Year'])
print("shape of the test target:{}".format(target_test.shape))

In [ ]:
features = data.columns.get_values()
drop_features = ["EPA", "Rating", "FE", "Cost", "Range", "CO2", "Guzzler", "MPG", "Smog", 'Unnamed', 'costs']
index = []
for i, feature in enumerate(features):
    flag = True
    if all(name not in feature for name in drop_features):
        index.append(i)

data = data.iloc[:, index]
data.shape

In [ ]:
list(data)

In [ ]:
# numerical features
float_data = list(data.select_dtypes(include=['float64']).columns)
# categorical features
object_data = list(data.select_dtypes(include=['object']).columns)

In [ ]:
float_data

In [ ]:
object_data

### Fill Missing Values

In [ ]:
data.isnull().sum()

We could see that column 'Max Ethanol % - Gasoline','Unique Label?','Label Recalc?','Var Valve Timing Desc' have missing values, then we should only fill NAs in these column

In [ ]:
#data[['Max Ethanol % - Gasoline','Unique Label?','Label Recalc?','Var Valve Timing Desc']]

'Max Ethanol % - Gasoline' contains numerical values; 'Unique Label?','Label Recalc?','Var Valve Timing Desc' contain string values

In [ ]:
imputer = Imputer(missing_values = 'NaN', strategy = 'mean', axis = 0)
data[['Max Ethanol % - Gasoline']] = imputer.fit_transform(data[['Max Ethanol % - Gasoline']])

In [ ]:
data['Unique Label?'] = data['Unique Label?'].fillna(data['Unique Label?'].value_counts().index[0])
data[['Label Recalc?']] = data['Label Recalc?'].fillna(data['Label Recalc?'].value_counts().index[0])
data[['Var Valve Timing Desc']] = data['Var Valve Timing Desc'].fillna(data['Var Valve Timing Desc'].value_counts().index[0])

## Linear Model

### Split Training and Validation

In [ ]:
dat = pd.get_dummies(data)

In [ ]:
#display(dat)

In [ ]:
dataval = dat[dat['Model Year']!=2018]
dataval = dataval.drop(columns = ['Model Year','Release Date'])
dataval.shape

In [ ]:
data_test = dat[data['Model Year']==2018]
data_test = data_test.drop(columns = ['Model Year','Release Date'])
data_test.shape

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(dataval, target_train, test_size = 0.25, random_state = 5)

In [ ]:
ridge = Ridge().fit(X_train, y_train)
print("Training set score: {:.2f}".format(ridge.score(X_train, y_train)))
print("Validation set score: {:.2f}".format(ridge.score(X_valid, y_valid)))
print("Test set score: {:.2f}".format(ridge.score(data_test, target_test)))

The score is quite good, we now try to find the best parameter using gridsearch

In [ ]:
param_grid = {'alpha' : np.logspace(-3,3,100)}
g_Ridge = GridSearchCV(Ridge(), param_grid, cv=10, return_train_score=True)
g_Ridge.fit(X_train, y_train)
print("Best parameter:{}".format(g_Ridge.best_params_))

Using the best parameter to fit the linear model

In [ ]:
ridge = Ridge(alpha=0.30538555088334157).fit(X_train, y_train)
print("Training set score: {:.2f}".format(ridge.score(X_train, y_train)))
print("Validation set score: {:.2f}".format(ridge.score(X_valid, y_valid)))
print("Test set score: {:.2f}".format(ridge.score(data_test, target_test)))

# Task 2: Feature Engineering

## Polynomial Features

In [ ]:
poly = PolynomialFeatures()
X_train_poly = poly.fit_transform(X_train)
X_train_poly.shape

In [ ]:
ridge_poly = Ridge().fit(X_train_poly, y_train)
print("Training set score: {:.2f}".format(0ridge_poly.score(X_valid_poly, y_valid))))

# Task 3: Any models

* We try two regression models: DecisionTreeRegressor, RandomForestRegressor. The RandomForestRegressor model is the best.

## Decision Tree

In [ ]:
param_grid = {'max_depth': range(1, dataval.shape[1], 100)}
gridDT = GridSearchCV(DecisionTreeRegressor(random_state = 0), param_grid = param_grid, cv = 10)
gridDT.fit(X_train, y_train)

In [ ]:
print(gridDT.best_estimator_, gridDT.best_score_)

In [ ]:
tree = DecisionTreeRegressor(random_state = 0, min_samples_split=2, max_features = 1601)
tree.fit(X_train, y_train)
print("Training set score: {:.2f}".format(tree.score(X_train, y_train)))
print("Validation set score: {:.2f}".format(tree.score(X_valid, y_valid)))
print("Test set score: {:.2f}".format(tree.score(data_test, target_test)))

## Random Forest

In [ ]:
param_grid = {'max_depth': range(1, dataval.shape[1], 100)}
gridRF = GridSearchCV(RandomForestRegressor(random_state = 0), param_grid = param_grid, cv = 10)
gridRF.fit(X_train, y_train)

In [ ]:
print(gridRF.best_estimator_, gridRF.best_score_)

In [ ]:
rf = RandomForestRegressor(max_depth=101, max_features = 'auto', oob_score = False,n_estimators = 10, random_state = 0)
rf.fit(X_train, y_train)
print("Training set score: {:.2f}".format(rf.score(X_train, y_train)))
print("Validation set score: {:.2f}".format(rf.score(X_valid, y_valid)))
print("Test set score: {:.2f}".format(rf.score(data_test, target_test)))

# Task 4: Feature Selections

## The most 10 influential features for Random Forest model are as follows
      1 Fuel Usage Desc - Conventional Fuel_Diesel, ultra low sulfur (15 ppm, maximum)
      2 Oil Viscosity_SAE 0W-20 API SN
      3 Stop/Start System (Engine Management System)  Description_Yes
      4 Index (Model Type Index)
      5 Air Aspiration Method Desc_Naturally Aspirated
      6 Stop/Start System (Engine Management System) Code_Y
      7 # Gears
      8 Trans Desc_Continuously Variable
      9 Carline Class
      10 Eng Displ

In [ ]:
select_best = np.argsort(rf.feature_importances_)[-10:]
best_importances = rf.feature_importances_[select_best]

In [ ]:
for rank, feature in enumerate(dataval.columns[select_best]):
    print(rank+1, feature)

In [ ]:
plt.figure()
plt.title("RandomForest Top 10")
plt.bar(range(10), best_importances, color = "g")
plt.xticks(range(10), dataval.columns[select_best], rotation = 90)
plt.show()

## The features of zero importance score could be removed without decrease in performance.
      There are 541 features of 0 importance score.

In [ ]:
sum(rf.feature_importances_==0)

## Remove irrelevant features for random forest
    We remove 541 irrelevant features. The test score is the same as our previous score.

In [ ]:
select = np.argsort(rf.feature_importances_)[541:]
X_train_select, X_valid_select = X_train.iloc[:, select], X_valid.iloc[:, select]
data_test_select = data_test.iloc[:, select]
rf = RandomForestRegressor(random_state=42).fit(X_train_select, y_train)
print("Training set score: ",rf.score(X_train_select, y_train))
print("Testing set score: ",rf.score(data_test_select, target_test))